This page is dedicated to experiment with Bivariate MVN distribution with anisotropic variance, with AR(1) rotation.  
[10, 1], [100, 1], [1000, 1], [100, 10], [1000, 10], [1000, 100]

In [1]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.2/188.2 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 MB 27.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.


### **Package Import and Other Setups**

In [1]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [3]:
drive.mount('/content/drive')
utility_link = '/content/drive/MyDrive/JHU Stuff/Capstone/Utility_Functions/BJX_Utility.py'
with open(utility_link) as f: exec(f.read())

Mounted at /content/drive


### **Hyperparam Setups**

In [4]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [5]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

In [6]:
def bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir):
  # mean array
  mu = jnp.full(num_dim, 0.0)
  # covariance matrix
  cov = jnp.array([
    [var1, rho*jnp.sqrt(var1*var2)],
    [rho*jnp.sqrt(var1*var2), var2]])
  target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
  init_step_size = 0.5
  def target_log_prob_fn(x):
    return target.log_prob(x)
  def initialize(shape, key):
    offset = 2
    return 10*random.normal(key, shape + (num_dim,)) + offset
  mean_benchmark = target.mean()
  var_benchmark = target.variance()

  #simulation part:
  Aniso_MSE_c_list = []
  Aniso_MSE_n_list = []
  Aniso_RHat_c_list = []
  Aniso_RHat_n_list = []

  base_key = random.PRNGKey(0)
  keys = random.split(base_key, repitition)
  for length in warmup_length:
    mem(f"Simulation Start")
    simulation(length,num_chains_short, num_super_chains,
               False,initialize, keys, target_log_prob_fn,init_step_size,
               repitition, Aniso_RHat_c_list,Aniso_MSE_c_list,mean_benchmark,var_benchmark)
    simulation(length,num_chains_short, num_super_chains,
               True,initialize, keys, target_log_prob_fn,init_step_size,
               repitition, Aniso_RHat_n_list,Aniso_MSE_n_list, mean_benchmark,var_benchmark)
  # save to dataframe and files
  MSE_c_df = pd.DataFrame(Aniso_MSE_c_list)
  R_Hat_c_df = pd.DataFrame(Aniso_RHat_c_list)
  MSE_n_df = pd.DataFrame(Aniso_MSE_n_list)
  R_Hat_n_df = pd.DataFrame(Aniso_RHat_n_list)

  # Use a descriptive parameter label
  label = f"var1_{var1:g}_var2_{var2:g}_rho_{rho:g}"

  MSE_c_df.to_pickle(f"{save_dir}/{label}_MSE_c.pkl")
  MSE_n_df.to_pickle(f"{save_dir}/{label}_MSE_n.pkl")
  R_Hat_c_df.to_pickle(f"{save_dir}/{label}_Rhat_c.pkl")
  R_Hat_n_df.to_pickle(f"{save_dir}/{label}_Rhat_n.pkl")

  del MSE_c_df, R_Hat_c_df, MSE_n_df, R_Hat_n_df
  gc.collect()
  print("Simulation Done, Data Saved!")


### **Experiment Part**

In [7]:
# Set up File Directory
save_dir = "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Unequal/"

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  10 & \rho*\sqrt{10} \\
  \rho*\sqrt{10} & 1 \\
\end{bmatrix}
$$

In [ ]:
var1 = 10.0
var2 = 1.0
num_dim = 2

In [ ]:
rho = 0.0
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1514.7 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.004469717852771282
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0022062421776354313
Simulation Start: 1787.5 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.000357325014192611
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0006795765948481858
Simulation Start: 1778.5 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00033610695390962064
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00038834859151393175
Simulation Start: 1773.9 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005124597810208797
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0003823752631433308
Simulation Start: 1775.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00023027586576063186
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00021931009541731328
Simulation Start: 1777.8 MB
Constraine

In [ ]:
rho = 0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1530.2 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.004098354838788509
Naive initialization. Warmup Length: 10; mean of MSE is: 0.003282728837803006
Simulation Start: 1794.9 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0005007347208447754
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00036426822771318257
Simulation Start: 1789.0 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0003499043523333967
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00022964733943808824
Simulation Start: 1797.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005074540385976434
Naive initialization. Warmup Length: 40; mean of MSE is: 0.00050457997713238
Simulation Start: 1807.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0002597089915070683
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00027728540590032935
Simulation Start: 1783.5 MB
Constrained i

In [ ]:
rho = 0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1795.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.006227622739970684
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0033075734972953796
Simulation Start: 1809.4 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0004271937650628388
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0004775053239427507
Simulation Start: 1801.7 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0003619732742663473
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00037748453905805945
Simulation Start: 1801.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004851265694014728
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0006566147785633802
Simulation Start: 1794.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0003321210388094187
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00033850959152914584
Simulation Start: 1816.0 MB
Constrained

In [ ]:
rho = 0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1803.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.054011035710573196
Naive initialization. Warmup Length: 10; mean of MSE is: 0.030001532286405563
Simulation Start: 1802.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.004544231574982405
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0008594624814577401
Simulation Start: 1809.1 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00040671491296961904
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0004833228013012558
Simulation Start: 1803.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0007081940420903265
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007017349125817418
Simulation Start: 1794.2 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0002561394067015499
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00028592912713065743
Simulation Start: 1792.8 MB
Constrained i

In [ ]:
rho = -0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

In [ ]:
rho = -0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

In [ ]:
rho = -0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  100 & \rho*\sqrt{100} \\
  \rho*\sqrt{100} & 1 \\
\end{bmatrix}
$$

In [ ]:
var1 = 100.0
var2 = 1.0
num_dim = 2

In [ ]:
rho = 0.0
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1809.6 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0317879281938076
Naive initialization. Warmup Length: 10; mean of MSE is: 0.015650173649191856
Simulation Start: 1803.4 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.02011977508664131
Naive initialization. Warmup Length: 20; mean of MSE is: 0.009139574132859707
Simulation Start: 1818.8 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.010543711483478546
Naive initialization. Warmup Length: 30; mean of MSE is: 0.005190025549381971
Simulation Start: 1795.1 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0077559067867696285
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0037758543621748686
Simulation Start: 1834.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.005322814919054508
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0020760896150022745
Simulation Start: 1804.5 MB
Constrained initializa

In [ ]:
rho = 0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1812.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.034680355340242386
Naive initialization. Warmup Length: 10; mean of MSE is: 0.01692797988653183
Simulation Start: 1831.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.020139383152127266
Naive initialization. Warmup Length: 20; mean of MSE is: 0.009930877014994621
Simulation Start: 1802.5 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.01091712899506092
Naive initialization. Warmup Length: 30; mean of MSE is: 0.006188061088323593
Simulation Start: 1819.8 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.007430369965732098
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0039975945837795734
Simulation Start: 1803.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.005130397155880928
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0023509457241743803
Simulation Start: 1818.8 MB
Constrained initializa

In [ ]:
rho = 0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1811.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.04409707710146904
Naive initialization. Warmup Length: 10; mean of MSE is: 0.02258431352674961
Simulation Start: 1804.1 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.028507057577371597
Naive initialization. Warmup Length: 20; mean of MSE is: 0.013943711295723915
Simulation Start: 1823.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.015582035295665264
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0073662749491631985
Simulation Start: 1796.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.010950724594295025
Naive initialization. Warmup Length: 40; mean of MSE is: 0.005779654253274202
Simulation Start: 1797.1 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00784970261156559
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0036009312607347965
Simulation Start: 1798.3 MB
Constrained initializat

In [ ]:
rho = 0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1805.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.06608114391565323
Naive initialization. Warmup Length: 10; mean of MSE is: 0.03544051572680473
Simulation Start: 1802.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0434570387005806
Naive initialization. Warmup Length: 20; mean of MSE is: 0.02445645071566105
Simulation Start: 1796.8 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.025936473160982132
Naive initialization. Warmup Length: 30; mean of MSE is: 0.01604526862502098
Simulation Start: 1803.8 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.021018553525209427
Naive initialization. Warmup Length: 40; mean of MSE is: 0.012580360285937786
Simulation Start: 1797.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.01500202901661396
Naive initialization. Warmup Length: 50; mean of MSE is: 0.008048688992857933
Simulation Start: 1812.0 MB
Constrained initialization. W

In [ ]:
rho = -0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

In [ ]:
rho = -0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

In [ ]:
rho = -0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  1000 & \rho*\sqrt{1000} \\
  \rho*\sqrt{1000} & 1 \\
\end{bmatrix}
$$

In [ ]:
var1 = 1000.0
var2 = 1.0
num_dim = 2

In [ ]:
rho = 0.0
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1821.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.005450294353067875
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0031984716188162565
Simulation Start: 1820.9 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.004784483462572098
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0024195050355046988
Simulation Start: 1814.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.004007960204035044
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0021109101362526417
Simulation Start: 1813.1 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0040201107040047646
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0019726043101400137
Simulation Start: 1805.3 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.004040105734020472
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0020336215384304523
Simulation Start: 1805.6 MB
Constrained init

In [ ]:
rho = 0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1814.7 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.006091711111366749
Naive initialization. Warmup Length: 10; mean of MSE is: 0.002669030800461769
Simulation Start: 1805.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.005190282128751278
Naive initialization. Warmup Length: 20; mean of MSE is: 0.002453476656228304
Simulation Start: 1811.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.004744418431073427
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0022251622285693884
Simulation Start: 1820.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.004543748218566179
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0020206174813210964
Simulation Start: 1806.5 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0042165061458945274
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0020015037152916193
Simulation Start: 1820.0 MB
Constrained initia

In [ ]:
rho = 0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1798.1 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.006648194044828415
Naive initialization. Warmup Length: 10; mean of MSE is: 0.003223010804504156
Simulation Start: 1819.2 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.006401888094842434
Naive initialization. Warmup Length: 20; mean of MSE is: 0.003182787448167801
Simulation Start: 1813.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0057455082423985004
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0025704256258904934
Simulation Start: 1804.3 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.005348305683583021
Naive initialization. Warmup Length: 40; mean of MSE is: 0.002765081822872162
Simulation Start: 1813.3 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0056443787179887295
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0023036678321659565
Simulation Start: 1804.2 MB
Constrained initia

In [ ]:
rho = 0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1804.8 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.009064007550477982
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0042946720495820045
Simulation Start: 1797.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.008488542400300503
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0038000866770744324
Simulation Start: 1811.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.007595055736601353
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0034272894263267517
Simulation Start: 1806.3 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.007697436958551407
Naive initialization. Warmup Length: 40; mean of MSE is: 0.003751461859792471
Simulation Start: 1813.3 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.007715785410255194
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0035890855360776186
Simulation Start: 1805.6 MB
Constrained initia

In [ ]:
rho = -0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

In [ ]:
rho = -0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1804.8 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.009064007550477982
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0042946720495820045
Simulation Start: 1797.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.008488542400300503
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0038000866770744324
Simulation Start: 1811.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.007595055736601353
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0034272894263267517
Simulation Start: 1806.3 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.007697436958551407
Naive initialization. Warmup Length: 40; mean of MSE is: 0.003751461859792471
Simulation Start: 1813.3 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.007715785410255194
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0035890855360776186
Simulation Start: 1805.6 MB
Constrained initia

In [ ]:
rho = -0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1804.8 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.009064007550477982
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0042946720495820045
Simulation Start: 1797.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.008488542400300503
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0038000866770744324
Simulation Start: 1811.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.007595055736601353
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0034272894263267517
Simulation Start: 1806.3 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.007697436958551407
Naive initialization. Warmup Length: 40; mean of MSE is: 0.003751461859792471
Simulation Start: 1813.3 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.007715785410255194
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0035890855360776186
Simulation Start: 1805.6 MB
Constrained initia

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  100 & \rho*\sqrt{100 * 10} \\
  \rho*\sqrt{100 * 10} & 10\\
\end{bmatrix}
$$

In [ ]:
var1 = 100.0
var2 = 10.0
num_dim = 2

In [8]:
rho = 0.0
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1516.7 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.015860484912991524
Naive initialization. Warmup Length: 10; mean of MSE is: 0.00827255193144083
Simulation Start: 1774.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.004421628080308437
Naive initialization. Warmup Length: 20; mean of MSE is: 0.002724024932831526
Simulation Start: 1771.5 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.001353184343315661
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0012479504803195596
Simulation Start: 1770.2 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0010228720493614674
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0008355122990906239
Simulation Start: 1789.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0005828562425449491
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00047854427248239517
Simulation Start: 1779.3 MB
Constrained initi

In [9]:
rho = 0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                          warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1776.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.017437689006328583
Naive initialization. Warmup Length: 10; mean of MSE is: 0.01146242581307888
Simulation Start: 1796.1 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.005065429024398327
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0037633455358445644
Simulation Start: 1810.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0015018952544778585
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0014093792997300625
Simulation Start: 1804.6 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0012355168582871556
Naive initialization. Warmup Length: 40; mean of MSE is: 0.001071203383617103
Simulation Start: 1789.5 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0006788434693589807
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0004859850159846246
Simulation Start: 1804.9 MB
Constrained initi

In [10]:
rho = 0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1787.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.03239466995000839
Naive initialization. Warmup Length: 10; mean of MSE is: 0.019341902807354927
Simulation Start: 1811.1 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.015360161662101746
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00941370613873005
Simulation Start: 1796.7 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.005216495133936405
Naive initialization. Warmup Length: 30; mean of MSE is: 0.004359428770840168
Simulation Start: 1791.2 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.003371547209098935
Naive initialization. Warmup Length: 40; mean of MSE is: 0.001962520182132721
Simulation Start: 1788.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.001426447881385684
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0006707143038511276
Simulation Start: 1797.6 MB
Constrained initializat

In [11]:
rho = 0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1785.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.06082697957754135
Naive initialization. Warmup Length: 10; mean of MSE is: 0.03520612418651581
Simulation Start: 1791.1 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.034177426248788834
Naive initialization. Warmup Length: 20; mean of MSE is: 0.019246462732553482
Simulation Start: 1803.0 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.019196610897779465
Naive initialization. Warmup Length: 30; mean of MSE is: 0.010758372023701668
Simulation Start: 1804.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.01209956593811512
Naive initialization. Warmup Length: 40; mean of MSE is: 0.006448439322412014
Simulation Start: 1797.1 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.007277282886207104
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0029708147048950195
Simulation Start: 1783.5 MB
Constrained initializati

In [ ]:
rho = -0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

In [ ]:
rho = -0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

In [ ]:
rho = -0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  1000 & \rho*\sqrt{1000 * 10} \\
  \rho*\sqrt{1000 * 10} & 10\\
\end{bmatrix}
$$

In [ ]:
var1 = 1000.0
var2 = 10.0
num_dim = 2

In [12]:
rho = 0.0
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1782.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.004497685935348272
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0024343656841665506
Simulation Start: 1789.0 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.004018538165837526
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0020417538471519947
Simulation Start: 1799.8 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.003400291781872511
Naive initialization. Warmup Length: 30; mean of MSE is: 0.001986522926017642
Simulation Start: 1804.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0030840090475976467
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0016836676513776183
Simulation Start: 1799.3 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.002862029243260622
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0013999146176502109
Simulation Start: 1798.8 MB
Constrained initi

In [13]:
rho = 0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1790.1 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.004888255149126053
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0027305432595312595
Simulation Start: 1790.7 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.004441668279469013
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0024499259889125824
Simulation Start: 1797.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0036863000132143497
Naive initialization. Warmup Length: 30; mean of MSE is: 0.002423106227070093
Simulation Start: 1799.2 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0034635260235518217
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0021603009663522243
Simulation Start: 1800.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0031907563097774982
Naive initialization. Warmup Length: 50; mean of MSE is: 0.001592232845723629
Simulation Start: 1792.0 MB
Constrained init

In [14]:
rho = 0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1791.6 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.006164113990962505
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0035341077018529177
Simulation Start: 1797.6 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.005623535718768835
Naive initialization. Warmup Length: 20; mean of MSE is: 0.003079683519899845
Simulation Start: 1791.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.005054747220128775
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0027733894530683756
Simulation Start: 1796.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.00467638811096549
Naive initialization. Warmup Length: 40; mean of MSE is: 0.002636548364534974
Simulation Start: 1797.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.004477728623896837
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0019196392968297005
Simulation Start: 1791.4 MB
Constrained initiali

In [15]:
rho = 0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1790.6 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.008797520771622658
Naive initialization. Warmup Length: 10; mean of MSE is: 0.004670483060181141
Simulation Start: 1812.9 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.008191095665097237
Naive initialization. Warmup Length: 20; mean of MSE is: 0.004019678104668856
Simulation Start: 1797.8 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.007454474922269583
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0043010166846215725
Simulation Start: 1798.3 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.007170782424509525
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0037326719611883163
Simulation Start: 1796.0 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.007188889198005199
Naive initialization. Warmup Length: 50; mean of MSE is: 0.003320434596389532
Simulation Start: 1792.0 MB
Constrained initiali

In [ ]:
rho = -0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

In [ ]:
rho = -0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

In [ ]:
rho = -0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  1000 & \rho*\sqrt{1000 * 100} \\
  \rho*\sqrt{1000 * 100} & 100\\
\end{bmatrix}
$$

In [ ]:
var1 = 1000.0
var2 = 100.0
num_dim = 2

In [16]:
rho = 0.0
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1786.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0021235088352113962
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0015800498658791184
Simulation Start: 1798.1 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0007485660607926548
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0005277118762023747
Simulation Start: 1793.0 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.000822228379547596
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0007900732452981174
Simulation Start: 1790.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0011410716688260436
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0008468645391985774
Simulation Start: 1798.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0005809358553960919
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0004209657199680805
Simulation Start: 1806.9 MB
Constrained i

In [17]:
rho = 0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1799.0 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.002854225691407919
Naive initialization. Warmup Length: 10; mean of MSE is: 0.002004563342779875
Simulation Start: 1807.7 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0008812196319922805
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0007651500636711717
Simulation Start: 1815.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0007742383750155568
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0008455677889287472
Simulation Start: 1828.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0007819395395927131
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0010583748808130622
Simulation Start: 1813.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.000491972197778523
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005337136099115014
Simulation Start: 1799.3 MB
Constrained ini

In [18]:
rho = 0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1792.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0049239471554756165
Naive initialization. Warmup Length: 10; mean of MSE is: 0.003412872552871704
Simulation Start: 1792.1 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.002616229699924588
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0015948174986988306
Simulation Start: 1798.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.001176465186290443
Naive initialization. Warmup Length: 30; mean of MSE is: 0.001211461960338056
Simulation Start: 1799.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0011985085438936949
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0011392036685720086
Simulation Start: 1792.3 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.000841400760691613
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0004286226467229426
Simulation Start: 1792.3 MB
Constrained initi

In [19]:
rho = 0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

Simulation Start: 1792.7 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.00815401691943407
Naive initialization. Warmup Length: 10; mean of MSE is: 0.005344565492123365
Simulation Start: 1801.5 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.006859864108264446
Naive initialization. Warmup Length: 20; mean of MSE is: 0.004414910916239023
Simulation Start: 1789.7 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.005240407772362232
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0036504454910755157
Simulation Start: 1793.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.004668055102229118
Naive initialization. Warmup Length: 40; mean of MSE is: 0.002984229940921068
Simulation Start: 1808.3 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.004099416080862284
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0017484743148088455
Simulation Start: 1785.8 MB
Constrained initializ

In [ ]:
rho = -0.3
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

In [ ]:
rho = -0.6
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

In [ ]:
rho = -0.9
bivariate_MVN_sampler(var1, var2, rho, num_dim, repitition,
                      warmup_length, num_chains_short, num_super_chains,save_dir)

#### **Autologout**

In [20]:
from google.colab import runtime
runtime.unassign()